# Prompt Engineering Bench

Notebook version of `prompt_engineering_bench.py`, organized by blocks so each stage can be executed step by step.

In [1]:
import os
import re
import time

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

c:\Users\mati1\Desktop\Repositorios\Agentic-Routine-RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## 1. Setup inicial

Se inicializa el modelo, las embeddings, la base vectorial y el retriever.

In [2]:
llm = ChatGroq(
    model_name="qwen/qwen3-32b",
    temperature=0.1,
    max_tokens=4000,
    groq_api_key=os.getenv("GROQ_API_KEY"),
)

embeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")
db = Chroma(persist_directory="chromadb_storage", embedding_function=embeddings)
retriever = db.as_retriever(search_kwargs={"k": 2})

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2211.75it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 2. Utilidades

Funciones auxiliares para formatear contexto y limpiar la salida del modelo.

In [3]:
def format_docs(docs):
    return "\n\n".join([
        f"Fuente: {d.metadata.get('source')}\nContenido: {d.page_content[:800]}"
        for d in docs
    ])


def clean_qwen_output(text):
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

## 3. Pregunta de prueba

La misma pregunta usada en el script original.

In [4]:
question = """
Hola, soy principiante y quiero aumentar la masa muscular de mi tren superior y piernas 
entrenando en mi casa 3 veces por semana. Estrictamente NO tengo pesas ni mancuernas, 
solo puedo usar mi peso corporal (calistenia) y una silla. 
Además, me duele un poco la rodilla derecha al flectarla mucho. 
¿Me puedes armar una rutina y explicarme cuántas series y descansos necesito según la ciencia?
"""

## 4. Estrategia 1: Zero-shot

Prompt base con recuperación de contexto y respuesta directa.

In [5]:
template_zs = """
Actúa como un entrenador personal experto basado estrictamente en evidencia científica deportiva.
Tu objetivo es diseñar rutinas de entrenamiento utilizando EXCLUSIVAMENTE el contexto proporcionado.

Reglas estrictas:
1. Usa solo el equipamiento que tiene el usuario.
2. Estructura en 3 partes: Volumen, Rutina, Justificación.
3. No des consejos médicos. Si hay dolor, da una advertencia general.

CONTEXTO: {context}
PREGUNTA: {question}
RESPUESTA:
"""
prompt_zs = ChatPromptTemplate.from_template(template_zs)
chain_zs = {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt_zs | llm | StrOutputParser()

In [9]:
print("\n1️⃣ TEST ZERO-SHOT:")
print(clean_qwen_output(chain_zs.invoke(question)))


1️⃣ TEST ZERO-SHOT:
**Volumen:**  
- **Series:** 3 por ejercicio.  
- **Repeticiones:** 10-15 para ejercicios de fuerza; 30-60 segundos para ejercicios isométricos (plank).  
- **Descanso:** 60 segundos entre series.  

---

**Rutina:**  
**Día 1: Tren superior (Push)**  
1. **Flexiones de pecho (Push-ups):** 3x10-15.  
   - *Modificación:* Si el suelo es difícil, realiza flexiones en la pared o con las manos elevadas en una silla.  
2. **Flexiones de tríceps en silla (Dips):** 3x8-12.  
   - *Técnica:* Mantén las piernas extendidas y apóyate en la silla con las manos.  
3. **Plank de codos:** 3x30-60 segundos.  
   - *Progresión:* Eleva una pierna o brazo al final de cada serie.  

**Día 2: Tren inferior (Piernas y glúteos)**  
1. **Sentadillas (Squats):** 3x10-15.  
   - *Modificación:* Reduce la profundidad si sientes dolor en la rodilla derecha.  
2. **Puente de glúteos (Glute Bridges):** 3x15-20.  
   - *Progresión:* Levanta una pierna durante la contracción.  
3. **Subidas de pi

## 5. Estrategia 2: Few-shot

Incluye ejemplos de comportamiento esperado para controlar mejor la salida.

In [6]:
template_fs = """
Actúa como un entrenador personal experto basado estrictamente en evidencia científica deportiva.
Tu objetivo es diseñar rutinas de entrenamiento utilizando EXCLUSIVAMENTE el contexto proporcionado.

Reglas estrictas de comportamiento:
1. Usa solo el equipamiento que tiene el usuario. Si pide algo que no está en el contexto, indícalo amablemente.
2. Si el contexto indica un número de series, repeticiones o descansos, úsalos exactamente.
3. REGLA DE SALUD: Si el usuario menciona dolor o molestias, INICIA tu respuesta con un bloque "**⚠️ ADVERTENCIA DE SEGURIDAD:**" recomendando consultar a un médico, y adapta la rutina omitiendo ejercicios que involucren esa zona. Si NO hay dolor, omite esta advertencia.
4. ESTRUCTURA DE LA RUTINA: 
   - Si el usuario menciona una cantidad de días a la semana, agrupa los ejercicios por día (ej. **Día 1: Enfoque...**).
   - Si NO menciona días, haz una lista secuencial de los ejercicios (1, 2, 3...).
   - En AMBOS casos, CADA ejercicio debe incluir obligatoriamente 3 viñetas: *Músculos objetivos*, *Cómo hacerlo* y *Modificación* (Agregar solo por dolor o falta de equipo).
5. Estructura tu respuesta final en 3 partes claras: 1. Parámetros de Entrenamiento, 2. Rutina Propuesta, 3. Justificación Científica.

--- EJEMPLO 1 (Menciona DÍAS y SÍ tiene DOLOR) ---
Pregunta: "Quiero entrenar tren superior 2 días a la semana en casa. Tengo bandas elásticas y dolor de codo."
Contexto: "Hipertrofia con bandas: 3-4 series, 10-15 reps. Descanso 60s. Ejercicios: Remo con banda, Aperturas de pecho, Press (evitar con dolor articular en codos)."
Tu Respuesta:
**⚠️ ADVERTENCIA DE SEGURIDAD:** Como entrenador, te recomiendo que si el dolor de codo es persistente, suspendas el ejercicio y consultes a un médico. Esta rutina omite movimientos de empuje directo por precaución.

### **1. Parámetros de Entrenamiento:**
- **Series:** 3 a 4 por ejercicio.
- **Repeticiones:** 10 a 15.
- **Descanso:** 60 segundos.

### **2. Rutina Propuesta:**
**Día 1: Tren superior tracción**
1. **Remo con banda**
   - *Músculos objetivos:* Dorsales y bíceps.
   - *Cómo hacerlo:* Sentado en el suelo, pasar la banda por los pies y tirar hacia el ombligo.
   - *Modificación:* Mantener los codos pegados al cuerpo para no forzar la articulación.

**Día 2: Tren superior aislamiento**
1. **Aperturas de pecho con banda (Chest flys)**
   - *Músculos objetivos:* Pectorales.
   - *Cómo hacerlo:* De pie, anclar la banda en la espalda y juntar los brazos extendidos al frente.
   - *Modificación:* Rango de movimiento corto, manteniendo una leve flexión estática del codo.

### **3. Justificación Científica:**
Se excluyeron ejercicios de empuje multiarticulares para proteger el codo, respetando los volúmenes de hipertrofia del contexto.

---

--- EJEMPLO 2 (NO menciona días y SÍ tiene DOLOR) ---
Pregunta: "Quiero hacer piernas con peso corporal y una silla. Me duele la espalda baja al agacharme."
Contexto: "Ejercicios peso corporal: Zancadas, Puente de glúteo, Sentadillas (evitar con dolor lumbar). Volumen: 3 series de 12 reps."
Tu Respuesta:
**⚠️ ADVERTENCIA DE SEGURIDAD:** Si presentas dolor en la espalda baja, te aconsejo consultar con un profesional de la salud. La siguiente rutina omite flexiones profundas de tronco por precaución.

### **1. Parámetros de Entrenamiento:**
- **Series:** 3
- **Repeticiones:** 12
- **Descanso:** 60 a 90 segundos.

### **2. Rutina Propuesta:**
1. **Zancadas estáticas**
   - *Músculos objetivos:* Cuádriceps y glúteos.
   - *Cómo hacerlo:* Dar un paso largo y bajar la rodilla trasera hacia el suelo manteniendo la espalda recta.
   - *Modificación:* Usar el respaldo de la silla como apoyo para mantener el equilibrio y el torso recto.

### **3. Justificación Científica:**
Se seleccionaron ejercicios que minimizan la carga axial en la columna lumbar según la evidencia.
--------------------------------------------------

CONTEXTO CIENTÍFICO RECUPERADO:
{context}

PREGUNTA DEL USUARIO: 
{question}

TU RESPUESTA:
"""
prompt_fs = ChatPromptTemplate.from_template(template_fs)
chain_fs = {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt_fs | llm | StrOutputParser()

In [10]:
print("\n2️⃣ TEST FEW-SHOT:")
print(clean_qwen_output(chain_fs.invoke(question)))


2️⃣ TEST FEW-SHOT:
**⚠️ ADVERTENCIA DE SEGURIDAD:** Si el dolor en la rodilla derecha persiste o empeora, consulta a un médico antes de continuar. Esta rutina evita movimientos de flexión profunda en la rodilla afectada para prevenir agravamientos.

---

### **1. Parámetros de Entrenamiento:**
- **Series:** 3 por ejercicio.
- **Repeticiones:** 10 a 15 (ajustar según fatiga).
- **Descanso:** 60 segundos entre series.
- **Días:** 3 veces por semana (ej. Lunes, Miércoles, Viernes).

---

### **2. Rutina Propuesta:**

#### **Día 1: Tren superior (Empuje)**
1. **Flexiones de pecho (Push-ups)**
   - *Músculos objetivos:* Pectorales, tríceps, deltoides anteriores.
   - *Cómo hacerlo:* En posición de flexión, mantén el cuerpo recto y baja el pecho hacia el suelo.
   - *Modificación:* Realízalas apoyando las rodillas en el suelo si sientes fatiga en los hombros.

2. **Elevaciones de piernas (Leg Raises)**
   - *Músculos objetivos:* Recto abdominal.
   - *Cómo hacerlo:* Tumbado en el suelo, ele

## 6. Estrategia 3: Chain-of-Thought con guardrails

Esta versión expone el razonamiento intermedio tal como lo hace el script original.

In [7]:
template_cot = """
Actúa como un entrenador personal experto.
Antes de generar la rutina final, debes realizar un análisis interno de RESTRICCIONES (Chain-of-Thought).

Sigue este proceso de razonamiento EXACTO y muéstralo en tu respuesta:
1. [Análisis de Equipo]: ¿Qué tiene disponible y qué está prohibido?
2. [Análisis de Salud]: ¿Hay dolor? Si es así, ¿qué ejercicios del contexto debo omitir por precaución?
3. [Análisis de Días]: ¿El usuario mencionó una cantidad de días a la semana? (Esto determina si la rutina se agrupa por "Días" o es una lista secuencial).
4.[Métricas RAG]: ¿Qué volumen/descanso dice el contexto?
5. [Generación]: Crear la rutina final. Cada ejercicio DEBE incluir las viñetas: *Músculos objetivos*, *Cómo hacerlo* y *Modificación* (Agregar solo por dolor o falta de equipo). Si hubo dolor, incluye la ⚠️ ADVERTENCIA DE SEGURIDAD al inicio.

--- EJEMPLO DE RAZONAMIENTO (Few-Shot CoT) ---
PREGUNTA: "Tengo bandas elásticas, dolor de hombro. Quiero rutina de 2 días."
CONTEXTO: "3 series de 10 reps. Ejercicios: flexiones, remo con banda."

RAZONAMIENTO PASO A PASO Y RESPUESTA:
1. [Análisis de Equipo]: Tiene bandas. Prohibidas pesas.
2.[Análisis de Salud]: Dolor de hombro. Omitir flexiones.
3. [Análisis de Días]: Sí, mencionó 2 días. La rutina debe ir agrupada por "Día 1" y "Día 2".
4.[Métricas RAG]: 3 series de 10 repeticiones.
5. [Generación]:
**⚠️ ADVERTENCIA DE SEGURIDAD:** Por tu dolor de hombro, te sugiero consultar a un médico. He omitido ejercicios de empuje directo.

### **1. Parámetros de Entrenamiento:**
- **Series:** 3 a 4 por ejercicio.
- **Repeticiones:** 10 a 15.
- **Descanso:** 60 segundos.

### **2. Rutina Propuesta:**
**Día 1: Tren superior tracción**
1. **Remo con banda**
   - *Músculos objetivos:* Dorsales y bíceps.
   - *Cómo hacerlo:* Sentado en el suelo, pasar la banda por los pies y tirar hacia el ombligo.
   - *Modificación:* Mantener los codos pegados al cuerpo para no forzar la articulación.

**Día 2: Tren superior aislamiento**
1. **Aperturas de pecho con banda (Chest flys)**
   - *Músculos objetivos:* Pectorales.
   - *Cómo hacerlo:* De pie, anclar la banda en la espalda y juntar los brazos extendidos al frente.
   - *Modificación:* Rango de movimiento corto, manteniendo una leve flexión estática del codo.

### **3. Justificación Científica:**
Se excluyeron ejercicios de empuje multiarticulares para proteger el codo, respetando los volúmenes de hipertrofia del contexto.
--------------------------------------------------

CONTEXTO CIENTÍFICO RECUPERADO:
{context}

PREGUNTA DEL USUARIO: 
{question}

RAZONAMIENTO PASO A PASO Y RESPUESTA:
"""
prompt_cot = ChatPromptTemplate.from_template(template_cot)
chain_cot = {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt_cot | llm | StrOutputParser()

In [11]:
print("\n3️⃣ TEST CHAIN-OF-THOUGHT (CoT):")
print(chain_cot.invoke(question))


3️⃣ TEST CHAIN-OF-THOUGHT (CoT):
<think>
Okay, let's tackle this user's request. They want a workout routine for upper body and legs, 3 times a week at home with just bodyweight and a chair. They have knee pain on the right when flexing too much. 

First, equipment analysis: They don't have weights or dumbbells, only bodyweight and a chair. So all exercises must use those. 

Health analysis: Knee pain when flexing a lot. Need to avoid exercises that put stress on the knees, especially the right one. Squats and lunges might be problematic. Maybe modify those or suggest alternatives like wall sits or step-ups with a lower range of motion.

Days per week: They mentioned 3 days, so the routine should be split into Day 1, Day 2, Day 3. Maybe upper body and legs on different days.

Metrics from the context: The example had 3 sets of 10 reps. The user didn't specify, but since they're a beginner, following standard hypertrophy ranges (3-4 sets, 8-15 reps) makes sense. Rest periods around 60-

## 7. Estrategia 4: Self-consistency

Genera varias respuestas, las compara y usa un juez para decidir la mejor.

In [8]:
def run_self_consistency(pregunta):
    print("\n" + "=" * 50)
    print("EJECUTANDO SELF-CONSISTENCY (Votación de 3 rutas)...")
    print("=" * 50)

    respuestas = []
    llm_variant = ChatGroq(
        model_name="qwen/qwen3-32b",
        temperature=0.5,
        max_tokens=2000,
        groq_api_key=os.getenv("GROQ_API_KEY"),
    )
    chain_variant = {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt_cot | llm_variant | StrOutputParser()

    for i in range(3):
        print(f"   Generando opción de razonamiento {i + 1}...")
        try:
            res = clean_qwen_output(chain_variant.invoke(pregunta))
            respuestas.append(res)
            if i < 2:
                print("   ⏳ Esperando 20s para evitar Rate Limit (TPM)...")
                time.sleep(20)
        except Exception as e:
            print(f"   ⚠️ Error en muestra {i + 1}: {e}")
            continue

    if len(respuestas) < 2:
        return "Error: No se pudieron generar suficientes muestras para consistencia."

    print("\nEl Juez (Llama 3.3 70B) está evaluando la mejor opción...")

    llm_judge = ChatGroq(
        model_name="llama-3.3-70b-versatile",
        temperature=0.1,
        groq_api_key=os.getenv("GROQ_API_KEY"),
    )

    judge_template = """
    Actúa como un Auditor de Seguridad Deportiva. Revisa estas 3 propuestas de rutina para un usuario que dijo:
    "{question}"
    
    Criterios de auditoría (Orden de importancia):
    1. ¿Incluyó la Advertencia de Seguridad obligatoria sobre el dolor?
    2. ¿Respetó estrictamente la restricción de NO USAR PESAS?
    3. ¿Evitó ejercicios que flexionen mucho la rodilla (ej. sentadillas profundas o saltos)?
    4. ¿Mantuvo una estructura clara y profesional en la respuesta? Manteniendo la división en Parámetros, Rutina y Justificación.
    
    OPCIÓN 1:
    {op1}
    ---
    OPCIÓN 2:
    {op2}
    ---
    OPCIÓN 3:
    {op3}
    
    Analiza brevemente las 3 opciones e indica ÚNICAMENTE el NÚMERO de la mejor opción (ejemplo: "OPCIÓN 2"). No transcribas la rutina.
    """

    time.sleep(10)

    res_juez = llm_judge.invoke(
        judge_template.format(
            question=pregunta,
            op1=respuestas[0][:],
            op2=respuestas[1][:],
            op3=respuestas[2][:],
        )
    )
    print(f"\nVeredicto: {res_juez.content}")

    return respuestas[0]

In [12]:
print("\n4️⃣ TEST SELF-CONSISTENCY:")
print(run_self_consistency(question))


4️⃣ TEST SELF-CONSISTENCY:

EJECUTANDO SELF-CONSISTENCY (Votación de 3 rutas)...
   Generando opción de razonamiento 1...
   ⏳ Esperando 20s para evitar Rate Limit (TPM)...
   Generando opción de razonamiento 2...
   ⏳ Esperando 20s para evitar Rate Limit (TPM)...
   Generando opción de razonamiento 3...

El Juez (Llama 3.3 70B) está evaluando la mejor opción...

Veredicto: OPCIÓN 1
**RAZONAMIENTO PASO A PASO Y RESPUESTA:**  
1. [Análisis de Equipo]: Solo tiene silla y peso corporal. Prohibidas pesas, mancuernas y bandas.  
2. [Análisis de Salud]: Dolor en rodilla derecha. Omitir ejercicios de flexión profunda de rodilla (ej: sentadillas, zancadas).  
3. [Análisis de Días]: Sí, mencionó 3 días. Rutina agrupada por "Día 1", "Día 2" y "Día 3".  
4. [Métricas RAG]: 3 series de 10-15 repeticiones. Descanso de 60-90 segundos (ajustable por fatiga).  
5. [Generación]:  
**⚠️ ADVERTENCIA DE SEGURIDAD:** Por tu dolor en la rodilla, evita movimientos con flexión profunda. Si el dolor persiste,